#Transformed races data
- Read bronze races table
- Keep only the columns required for analytics (Drop url column)
- Standardise column names using snake_case (circuitId → circuit_id, raceName → race_name)
- Rename columns to make them more meaningful (date -> race_date)
- Remove duplicate records
- Transform values of columns race_name and locality to Title Case
- Write the transformed data to silver races table

In [0]:
%run ../00-common/01.environment.config


In [0]:
%python
bronze_table = f"{catalog_name}.{bronze_schema_name}.races"
silver_table = f"{catalog_name}.{silver_schema_name}.races"

#Read bronze races table

In [0]:
%python
races_df = spark.read.table(bronze_table)

In [0]:
%python
display(races_df)
       


#Keep only the columns required for analytics (Drop url column)

In [0]:
%python
 races_selected_df = races_df.select(
    "season",
    "round",
    "racename",
    "date",
    "circuitID",
    "ingestion_timestamp",
    "source_file"
)

#Standardise column names using snake_case (circuitId → circuit_id, raceName → race_name)
#Rename columns to make them more meaningful (date -> race_date)

In [0]:
%python
races_renamed_df = (races_selected_df.withColumnRenamed("circuitId", "circuit_id")
.withColumnRenamed("circuitId", "circuit_id")
.withColumnRenamed("racename", "race_name")
.withColumnRenamed("date", "race_date")
)

#Remove duplicate records

In [0]:
%python
races_distinct_df = races_renamed_df.dropDuplicates(["season","round"])

#Transform values of columns race_name to Title Case

In [0]:
%python
from pyspark.sql import functions as F
races_final_df = (races_distinct_df
                     .withColumn("race_name", F.initcap(F.col("race_name")))
                   

)

In [0]:
%python
display(races_final_df)

#Write the transformed data to silver races table

In [0]:
%python
(
    races_final_df
    .write
    .format("delta") 
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
%python
display(spark.table(silver_table))